# Exercise 1 — Post-hoc XAI for Image Classification with CIFAR-10

**PCS956 • Module C • Week 2**

## Aim

In this exercise you will train a convolutional neural network (CNN) on the **CIFAR-10 image dataset** and analyse one correctly classified test image using three **model-specific post-hoc XAI methods**:

- **Saliency**
- **Integrated Gradients (IG)**
- **Grad-CAM**



## Learning objectives

After completing the exercise, you should be able to:

- train and evaluate a CNN before interpreting its explanations;
- identify a suitable target output for gradient-based XAI;
- compute and interpret a saliency map;
- compute Integrated Gradients relative to a specified baseline;
- compute a Grad-CAM localisation map from an internal convolutional layer;
- compare explanation methods without treating visual agreement as proof of faithfulness;
- distinguish model attribution or sensitivity from causal importance.

## 1. Load CIFAR-10

CIFAR-10 contains **60,000 colour images** of size **32 × 32 pixels** in **10 classes**. The standard split contains 50,000 training images and 10,000 test images.

The classes are:

`airplane`, `automobile`, `bird`, `cat`, `deer`, `dog`, `frog`, `horse`, `ship`, `truck`.

We use the standard test set only for **final model evaluation and post-hoc explanation**, after the model has been trained on the training data.

In [ ]:
# Uncomment in Google Colab if TensorFlow is not available:
!pip -q install tensorflow

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10

RANDOM_STATE = 42
tf.random.set_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

CLASS_NAMES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

(x_train, y_train), (x_test, y_test) = cifar10.load_data()

y_train = y_train.squeeze()
y_test = y_test.squeeze()

print("Training images:", x_train.shape)
print("Test images:", x_test.shape)


### Task 1 — Inspect the dataset

Display several training images and their labels.

Then answer:

1. What is the shape of one image?
2. Are these images high resolution or low resolution?
3. Why might low spatial resolution affect interpretation of explanation maps?

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))

for i, ax in enumerate(axes.ravel()):
    ax.imshow(x_train[i])
    ax.set_title(CLASS_NAMES[y_train[i]])
    ax.axis("off")

plt.tight_layout()
plt.show()


## 2. Build and train a CNN

The network outputs **logits** rather than probabilities. This is useful because the gradient-based methods below will explain the selected class **logit** directly.

The softmax transformation is used only when probabilities are required for reporting predictions.

### Task 2

Train the model and report test accuracy before generating explanations.

In [ ]:
# Convert pixels to [0, 1]
x_train_f = x_train.astype("float32") / 255.0
x_test_f = x_test.astype("float32") / 255.0

model = models.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, 3, padding="same", activation="relu", name="conv1"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, padding="same", activation="relu", name="conv2"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, padding="same", activation="relu", name="conv3"),
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.30),
    layers.Dense(10, name="logits")
])

model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history = model.fit(
    x_train_f,
    y_train,
    validation_split=0.10,
    epochs=8,
    batch_size=128,
    verbose=1
)

test_loss, test_accuracy = model.evaluate(
    x_test_f,
    y_test,
    verbose=0
)

print(f"Test accuracy: {test_accuracy:.3f}")


### Performance note

The objective of this exercise is **explanation analysis rather than state-of-the-art CIFAR-10 performance**. Model performance should nevertheless be considered when interpreting the explanations.


### Scientific checkpoint

Explanation quality should not be assessed in isolation from predictive behaviour.

Before interpreting an explanation, verify that:

- the model has learned a non-trivial predictive function;
- the selected observation has a valid prediction;
- the explanation target is stated explicitly.

In this notebook, **Saliency, Integrated Gradients, and Grad-CAM all target the same selected class logit**.

## 3. Select a correctly classified test image

To avoid explaining an obviously incorrect prediction in the main exercise, select the **first correctly classified test image**.

This does not mean that XAI should only be applied to correct predictions. Misclassified observations can also be informative, but the interpretation question is then different.

In [ ]:
test_logits = model.predict(x_test_f, verbose=0)
test_probabilities = tf.nn.softmax(test_logits, axis=1).numpy()
test_predictions = np.argmax(test_probabilities, axis=1)

correct_indices = np.where(test_predictions == y_test)[0]
instance_idx = int(correct_indices[0])

image = x_test_f[instance_idx:instance_idx + 1]
true_class = int(y_test[instance_idx])
predicted_class = int(test_predictions[instance_idx])
predicted_probability = float(test_probabilities[instance_idx, predicted_class])

print("Test index:", instance_idx)
print("True class:", CLASS_NAMES[true_class])
print("Predicted class:", CLASS_NAMES[predicted_class])
print(f"Predicted probability: {predicted_probability:.3f}")

plt.figure(figsize=(4, 4))
plt.imshow(x_test[instance_idx])
plt.title(
    f"True: {CLASS_NAMES[true_class]} | "
    f"Predicted: {CLASS_NAMES[predicted_class]}"
)
plt.axis("off")
plt.show()


### Interpretation note

A correct prediction can still be uncertain. The predicted probability of the selected observation should therefore be considered when interpreting its explanation.


## 4. Saliency — local input sensitivity

**Saliency maps** use gradients of a selected model score with respect to the input to show where small input changes would most strongly affect that score.

This characterises **local input sensitivity**. It does not establish causal importance.

### Task 3

Compute the saliency map for the predicted class.

In [ ]:
input_tensor = tf.convert_to_tensor(image)

with tf.GradientTape() as tape:
    tape.watch(input_tensor)
    logits = model(input_tensor, training=False)
    target_logit = logits[:, predicted_class]

gradients = tape.gradient(target_logit, input_tensor)

saliency = tf.reduce_max(
    tf.abs(gradients),
    axis=-1
)[0].numpy()

scale = np.percentile(saliency, 99)
saliency_display = np.clip(
    saliency / (scale + 1e-12),
    0,
    1
)

plt.figure(figsize=(5, 5))
plt.imshow(x_test[instance_idx])
plt.imshow(
    saliency_display,
    alpha=0.55,
    cmap="jet"
)
plt.title("Saliency — absolute local input sensitivity")
plt.axis("off")
plt.show()


### Discussion 1

1. Which spatial regions have high gradient magnitude?
2. Is the saliency map smooth or noisy?
3. What does a large absolute gradient mean mathematically?
4. Why does it **not** establish that the highlighted pixel caused the prediction?
5. What access to the model is required to compute this explanation?

## 5. Integrated Gradients — baseline-relative attribution

**Integrated Gradients (IG)** attributes the selected model output to the input relative to a chosen baseline by accumulating gradients along a path between the baseline and the input.

The baseline is part of the explanation and can affect the resulting attribution.

### Task 4

Compute Integrated Gradients for the same predicted class and inspect the numerical completeness check produced by the code.

In [ ]:
def integrated_gradients(
    model,
    input_tensor,
    target_class,
    baseline=None,
    steps=256
):
    if baseline is None:
        baseline = tf.zeros_like(input_tensor)

    alphas = tf.linspace(0.0, 1.0, steps + 1)

    interpolated = (
        baseline
        + alphas[:, None, None, None]
        * (input_tensor - baseline)
    )

    with tf.GradientTape() as tape:
        tape.watch(interpolated)
        logits = model(interpolated, training=False)
        target_logits = logits[:, target_class]

    grads = tape.gradient(
        target_logits,
        interpolated
    )

    # Trapezoidal approximation of the line integral
    trapezoids = (grads[:-1] + grads[1:]) / 2.0
    avg_grads = tf.reduce_mean(
        trapezoids,
        axis=0,
        keepdims=True
    )

    attributions = (
        input_tensor - baseline
    ) * avg_grads

    return attributions, baseline


ig, baseline = integrated_gradients(
    model=model,
    input_tensor=input_tensor,
    target_class=predicted_class,
    steps=256
)

# Visualise attribution magnitude across RGB channels
ig_map = tf.reduce_sum(
    tf.abs(ig),
    axis=-1
)[0].numpy()

scale = np.percentile(ig_map, 99)
ig_display = np.clip(
    ig_map / (scale + 1e-12),
    0,
    1
)

plt.figure(figsize=(5, 5))
plt.imshow(x_test[instance_idx])
plt.imshow(
    ig_display,
    alpha=0.55,
    cmap="jet"
)
plt.title("Integrated Gradients — attribution magnitude")
plt.axis("off")
plt.show()

input_logit = float(
    model(input_tensor, training=False)[0, predicted_class]
)
baseline_logit = float(
    model(baseline, training=False)[0, predicted_class]
)
ig_sum = float(tf.reduce_sum(ig))

difference = input_logit - baseline_logit
absolute_error = abs(difference - ig_sum)
relative_error = absolute_error / (abs(difference) + 1e-12)

print(f"Output difference: {difference:.6f}")
print(f"Sum of IG:        {ig_sum:.6f}")
print(f"Absolute error:   {absolute_error:.6f}")
print(f"Relative error:   {relative_error:.2%}")


### Discussion 2

1. How does the IG map differ from the saliency map?
2. Why is the baseline scientifically relevant?
3. What does the completeness check evaluate?
4. Why can numerical completeness be imperfect?
5. Would changing the baseline potentially change the attribution map?

## 6. Grad-CAM — class-discriminative spatial localisation

**Gradient-weighted Class Activation Mapping (Grad-CAM)** uses gradients of a selected class score with respect to internal convolutional feature maps to produce a coarse localisation map of image regions associated with that class.

### Task 5

Compute Grad-CAM for the same predicted class used in the previous methods.

In [ ]:
target_layer_name = "conv3"

grad_model = tf.keras.Model(
    inputs=model.inputs,
    outputs=[
        model.get_layer(target_layer_name).output,
        model.layers[-1].output
    ]
)

with tf.GradientTape() as tape:
    feature_maps, logits = grad_model(
        input_tensor,
        training=False
    )
    target_logit = logits[:, predicted_class]

gradients = tape.gradient(
    target_logit,
    feature_maps
)

weights = tf.reduce_mean(
    gradients,
    axis=(1, 2)
)

cam = tf.reduce_sum(
    feature_maps[0]
    * weights[0][None, None, :],
    axis=-1
)

cam = tf.nn.relu(cam)
cam = cam / (tf.reduce_max(cam) + 1e-12)

cam_resized = tf.image.resize(
    cam[..., None],
    (32, 32),
    method="bilinear"
)[..., 0].numpy()

plt.figure(figsize=(5, 5))
plt.imshow(x_test[instance_idx])
plt.imshow(
    cam_resized,
    alpha=0.55,
    cmap="jet"
)
plt.title("Grad-CAM — coarse class-discriminative localisation")
plt.axis("off")
plt.show()

### Discussion 3

1. Is Grad-CAM more spatially coherent than the saliency map?
2. Why is Grad-CAM described as **coarse localisation**?
3. What model-internal information does Grad-CAM require?
4. Why should a visually plausible Grad-CAM heatmap not be treated as proof that the model is faithful, robust, fair, or causally correct?

## 7. Compare the three explanations

Complete the table.

| Method | Explanation target | Model-internal information required | Reference-dependent? | Primary interpretation |
|---|---|---|---|---|
| Saliency |  |  |  |  |
| Integrated Gradients |  |  |  |  |
| Grad-CAM |  |  |  |  |

### Exercise — Interpret one explanation

Choose **one** of the three explanation methods:

- Saliency
- Integrated Gradients
- Grad-CAM

Write a short interpretation of the result in **3–4 sentences**.

Your interpretation should state:

1. what quantity the method is showing;
2. which image regions receive the strongest response;
3. one limitation of the explanation.



## References

- Krizhevsky, A. (2009). *Learning Multiple Layers of Features from Tiny Images.* Technical Report, University of Toronto. https://www.cs.toronto.edu/~kriz/learning-features-2009-TR.pdf

- Simonyan, K., Vedaldi, A., & Zisserman, A. (2014). *Deep Inside Convolutional Networks: Visualising Image Classification Models and Saliency Maps.* ICLR Workshop. https://arxiv.org/abs/1312.6034

- Sundararajan, M., Taly, A., & Yan, Q. (2017). *Axiomatic Attribution for Deep Networks.* Proceedings of the 34th International Conference on Machine Learning (ICML), 70, 3319–3328. https://proceedings.mlr.press/v70/sundararajan17a.html

- Selvaraju, R. R., Cogswell, M., Das, A., Vedantam, R., Parikh, D., & Batra, D. (2017). *Grad-CAM: Visual Explanations from Deep Networks via Gradient-Based Localization.* Proceedings of the IEEE International Conference on Computer Vision (ICCV), 618–626. https://doi.org/10.1109/ICCV.2017.74
